# MoneyPrinterTurbo GitHub Issues queue

This notebook starts only the GitHub Issues queue. It always checks out the current `Equilibriumpress/MoneyPrinterTurbo` fork, validates both tokens, starts the local API, and then starts the issue worker.


## 1. Install the current fork

Run this cell first, including when an older MoneyPrinterTurbo checkout already exists in the Colab runtime.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/MoneyPrinterTurbo")
REPO_URL = "https://github.com/Equilibriumpress/MoneyPrinterTurbo.git"
REPO_BRANCH = "main"

if (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"],
        check=True,
    )
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository")
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

worker_path = REPO_DIR / "scripts" / "github_issue_worker.py"
if not worker_path.is_file():
    origin = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"],
        text=True,
    ).strip()
    raise RuntimeError(
        f"Queue worker ontbreekt na update. Huidige origin: {origin}. "
        "Start een nieuwe Colab-runtime en voer deze cel opnieuw uit."
    )

os.chdir(REPO_DIR)
subprocess.run(
    ["python", "-m", "pip", "install", "-q", "uv", "pyngrok"],
    check=True,
)
subprocess.run(["uv", "python", "install", "3.11"], check=True)
subprocess.run(
    ["uv", "sync", "--frozen", "--python", "3.11"],
    check=True,
)

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()
origin = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"],
    text=True,
).strip()

print(f"Repository: {origin}")
print(f"Commit: {commit}")
print(f"Worker: {worker_path}")


## 2. Validate ngrok and GitHub

The tokens are read with hidden input. The GitHub test verifies the authenticated account and access to the target repository before starting MoneyPrinterTurbo.


In [ ]:
import json
from getpass import getpass
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

from pyngrok import ngrok

ngrok.kill()
ngrok_token = getpass("Enter the ngrok authentication token: ").strip()
if not ngrok_token:
    raise ValueError("An ngrok authentication token is required")
ngrok.set_auth_token(ngrok_token)
del ngrok_token

github_token = getpass("Enter the fine-grained GitHub token: ").strip()
if not github_token:
    raise ValueError("A GitHub token is required")

def github_get(path):
    request = Request(
        f"https://api.github.com{path}",
        headers={
            "Accept": "application/vnd.github+json",
            "Authorization": f"Bearer {github_token}",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "MoneyPrinterTurbo-Colab-Diagnostics",
        },
    )
    try:
        with urlopen(request, timeout=30) as response:
            return json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"GitHub token test failed with HTTP {exc.code}: {detail}"
        ) from exc
    except URLError as exc:
        raise RuntimeError(f"GitHub is niet bereikbaar: {exc}") from exc

profile = github_get("/user")
repository = github_get("/repos/Equilibriumpress/MoneyPrinterTurbo")
queued = github_get(
    "/repos/Equilibriumpress/MoneyPrinterTurbo/issues"
    "?state=open&labels=video-job&per_page=30"
)

print(f"GitHub account: {profile.get('login')}")
print(f"Repository access: {repository.get('full_name')}")
print(f"Queued video jobs: {len([item for item in queued if 'pull_request' not in item])}")
print("ngrok and GitHub authentication are valid.")


## 3. Start the API and queue

This cell clears stale processes, waits up to five minutes for the MoneyPrinterTurbo API, opens a temporary result tunnel, and starts the issue worker.


In [ ]:
import os
import subprocess
import time
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

API_PORT = 8080
API_LOG_PATH = Path("/content/moneyprinterturbo-api.log")
WORKER_LOG_PATH = Path("/content/moneyprinterturbo-github-worker.log")

for process_name in ("github_worker_proc", "api_proc"):
    previous_process = globals().get(process_name)
    if previous_process is not None and previous_process.poll() is None:
        previous_process.terminate()
        try:
            previous_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            previous_process.kill()
            previous_process.wait(timeout=5)

for log_name in ("github_worker_log", "api_log"):
    previous_log = globals().get(log_name)
    if previous_log is not None and not previous_log.closed:
        previous_log.close()

previous_api_tunnel = globals().get("api_tunnel")
if previous_api_tunnel is not None:
    try:
        ngrok.disconnect(previous_api_tunnel.public_url)
    except Exception:
        pass

subprocess.run(
    ["bash", "-lc", "pkill -f '[g]ithub_issue_worker.py' >/dev/null 2>&1 || true"],
    check=False,
)
subprocess.run(
    ["bash", "-lc", f"fuser -k {API_PORT}/tcp >/dev/null 2>&1 || true"],
    check=False,
)
time.sleep(1)

api_env = os.environ.copy()
api_env["PYTHONUNBUFFERED"] = "1"
api_log = API_LOG_PATH.open("w", encoding="utf-8")
api_proc = subprocess.Popen(
    ["uv", "run", "python", "-u", "main.py"],
    cwd=REPO_DIR,
    env=api_env,
    stdout=api_log,
    stderr=subprocess.STDOUT,
    text=True,
)

deadline = time.time() + 300
next_log_report = time.time() + 30
api_ready = False

while time.time() < deadline:
    try:
        with urlopen(
            f"http://127.0.0.1:{API_PORT}/openapi.json",
            timeout=2,
        ) as response:
            api_ready = response.status == 200
    except (URLError, TimeoutError):
        pass

    if api_ready or api_proc.poll() is not None:
        break

    if time.time() >= next_log_report:
        api_log.flush()
        recent = API_LOG_PATH.read_text(
            encoding="utf-8",
            errors="replace",
        )[-1200:]
        print("API is still starting. Latest log:")
        print(recent or "(no log output yet)")
        next_log_report = time.time() + 30

    time.sleep(2)

if not api_ready:
    api_log.flush()
    recent_log = API_LOG_PATH.read_text(
        encoding="utf-8",
        errors="replace",
    )[-12000:]
    raise RuntimeError(
        "MoneyPrinterTurbo API failed to start.\n"
        f"Process return code: {api_proc.poll()}\n"
        f"Recent log:\n{recent_log}"
    )

api_tunnel = ngrok.connect(
    addr=f"http://127.0.0.1:{API_PORT}",
    proto="http",
    bind_tls=True,
)

worker_env = os.environ.copy()
worker_env["PYTHONUNBUFFERED"] = "1"
worker_env["MPT_GITHUB_TOKEN"] = github_token
worker_env["MPT_GITHUB_REPOSITORY"] = "Equilibriumpress/MoneyPrinterTurbo"
worker_env["MPT_API_BASE"] = f"http://127.0.0.1:{API_PORT}/api/v1"
worker_env["MPT_PUBLIC_BASE"] = api_tunnel.public_url
worker_env["MPT_MAX_VIDEO_COUNT"] = "1"

github_worker_log = WORKER_LOG_PATH.open("w", encoding="utf-8")
github_worker_proc = subprocess.Popen(
    [
        "uv",
        "run",
        "python",
        "-u",
        "scripts/github_issue_worker.py",
        "--poll-seconds",
        "20",
    ],
    cwd=REPO_DIR,
    env=worker_env,
    stdout=github_worker_log,
    stderr=subprocess.STDOUT,
    text=True,
)

worker_deadline = time.time() + 30
worker_ready = False
while time.time() < worker_deadline:
    github_worker_log.flush()
    worker_text = WORKER_LOG_PATH.read_text(
        encoding="utf-8",
        errors="replace",
    )
    if " watches " in worker_text:
        worker_ready = True
        break
    if github_worker_proc.poll() is not None:
        break
    time.sleep(1)

if not worker_ready:
    github_worker_log.flush()
    recent_log = WORKER_LOG_PATH.read_text(
        encoding="utf-8",
        errors="replace",
    )[-12000:]
    raise RuntimeError(
        "GitHub worker failed to become ready.\n"
        f"Process return code: {github_worker_proc.poll()}\n"
        f"Recent log:\n{recent_log}"
    )

print("GitHub Issues queue is running.")
print(f"Temporary result URL: {api_tunnel.public_url}")
print(f"API docs: {api_tunnel.public_url}/docs")
print(f"Worker log: {WORKER_LOG_PATH}")
print(f"API log: {API_LOG_PATH}")


## 4. Inspect status

Run this cell when an issue stays on `video-job`. It prints both process states and the latest logs.


In [ ]:
print(f"API process return code: {api_proc.poll()}")
print(f"Worker process return code: {github_worker_proc.poll()}")

print("\nWorker log:\n")
print(
    WORKER_LOG_PATH.read_text(
        encoding="utf-8",
        errors="replace",
    )[-12000:]
)

print("\nAPI log:\n")
print(
    API_LOG_PATH.read_text(
        encoding="utf-8",
        errors="replace",
    )[-12000:]
)
